# 🤖 AI Job Market Global 2026 — Starter Notebook

Welcome! This notebook gives you a quick tour of the **AI Job Market Global 2026** dataset:  
5,773 real-time job postings collected from the **Adzuna** and **USAJobs** public APIs across 🇺🇸 🇬🇧 🇨🇦 🇦🇺 🇩🇪.

**What we'll cover:**
1. 📦 Load & inspect the data
2. 💰 Salary landscape by country
3. 🛠️ Most in-demand skills
4. 🌐 Remote vs onsite breakdown
5. 📈 Experience level distribution
6. 🔍 Salary prediction — baseline model

In [ ]:
# ── Install / import ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from collections import Counter

# ── Load dataset ──────────────────────────────────────────────────────────────
# On Kaggle the CSV will be at /kaggle/input/<dataset-slug>/ai_jobs_global.csv
import os
BASE = '/kaggle/input/ai-job-market-global-2026' if os.path.exists('/kaggle') else 'data'
df = pd.read_csv(f'{BASE}/ai_jobs_global.csv', encoding='utf-8', parse_dates=['posted_date'])

print(f"✅ Loaded  {len(df):,} rows  ×  {df.shape[1]} columns")
df.head(3)

## 📦 1. Dataset Overview

In [ ]:
# Quick stats overview
overview = pd.DataFrame({
    'dtype'      : df.dtypes,
    'non_null'   : df.notna().sum(),
    'null_%'     : (df.isna().mean() * 100).round(1),
    'unique'     : df.nunique(),
})

print(f"{'='*50}")
print(f"  Jobs total       : {len(df):,}")
print(f"  Countries        : {df['country'].nunique()}")
print(f"  Unique companies : {df['company'].nunique():,}")
print(f"  Date range       : {df['posted_date'].min().date()}  →  {df['posted_date'].max().date()}")
print(f"  Salary coverage  : {df['salary_min'].notna().mean()*100:.1f}% of rows have salary data")
print(f"{'='*50}")
display(overview)

## 💰 2. Salary Landscape by Country

All salaries are already normalised to **annual USD**. Let's see how compensation compares across markets.

In [ ]:
sal = df[df['salary_min'].notna() & (df['salary_min'] > 0)].copy()

# Summary table
summary = (
    sal.groupby('country')['salary_min']
    .agg(count='count', median='median', mean='mean', p25=lambda x: x.quantile(0.25), p75=lambda x: x.quantile(0.75))
    .round(0).astype(int)
    .sort_values('median', ascending=False)
    .reset_index()
)
summary.columns = ['Country', 'Jobs w/ Salary', 'Median USD', 'Mean USD', 'P25 USD', 'P75 USD']
display(summary)

# Box plot
fig = px.box(
    sal[sal['country'].isin(summary['Country'])],
    x='country', y='salary_min', color='country',
    color_discrete_sequence=px.colors.qualitative.Bold,
    title='Annual Salary Distribution by Country (USD — Min Reported)',
    labels={'salary_min': 'Salary (USD)', 'country': 'Country'},
    points='outliers',
    category_orders={'country': summary['Country'].tolist()},
)
fig.update_layout(
    showlegend=False, plot_bgcolor='white',
    yaxis=dict(gridcolor='#eeeeee', tickprefix='$', tickformat=',.0f'),
    title_font_size=17,
)
fig.show()

## 🛠️ 3. Most In-Demand Skills

The `required_skills` column contains comma-separated skills extracted from each job description.

In [ ]:
# Flatten comma-separated skills
all_skills = []
for entry in df['required_skills'].dropna():
    all_skills.extend([s.strip() for s in str(entry).split(',') if s.strip()])

skill_counts = pd.Series(Counter(all_skills)).sort_values(ascending=True).tail(20).reset_index()
skill_counts.columns = ['skill', 'count']
skill_counts['pct'] = (skill_counts['count'] / len(df) * 100).round(1)

fig = px.bar(
    skill_counts, x='count', y='skill', orientation='h',
    color='count', color_continuous_scale='Blues',
    title='Top 20 In-Demand AI/ML Skills (by job posting count)',
    labels={'count': 'Job Postings', 'skill': ''},
    text=skill_counts['pct'].astype(str) + '%',
)
fig.update_traces(textposition='outside')
fig.update_layout(
    plot_bgcolor='white', coloraxis_showscale=False,
    xaxis=dict(gridcolor='#eeeeee'), height=600, title_font_size=17,
)
fig.show()

print("\nTop 5 skills by % of all postings:")
print(skill_counts.tail(5)[['skill','pct']].iloc[::-1].to_string(index=False))

## 🌐 4. Remote vs Hybrid vs Onsite  &  📈 5. Experience Level Distribution